# Find Best Model


In [1]:
from typing import Callable
from types import SimpleNamespace
from collections import defaultdict
import os
import glob
import math
import numpy as np
import pandas as pd
import torch
import lightning as pl
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots


from tqdm import tqdm

from meta import load_meta
from model import MultiLayerLeakyReLUModel
from nearest_neighbour_model import (
    NearestNeighbourModel,
    NearestNeighboursInterpolationModel,
)

## Args


### Demo


In [2]:
# args = SimpleNamespace(
#     # io
#     logs_path="data/logs",
#     experiment_name="demo_experiment",
# )

### laser-pulse-shaping-astra-sim


In [3]:
# args = SimpleNamespace(
#     # io
#     data_path="data/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
#     logs_path="data/logs",
#     experiment_name="laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",  # use .old for previous version
# )

args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed",
)

In [4]:
rows: list[dict] = []

for run_path in glob.glob(os.path.join(args.logs_path, args.experiment_name, "*")):
    print(f"Loading meta from {run_path}")
    meta = load_meta(run_path)
    rows.append(meta)

table = pd.DataFrame(rows)

assert all(table["data_path"] == args.data_path)

table["max_width"] = table["hidden_layer_sizes"].apply(lambda x: max(x))
table["depth"] = table["hidden_layer_sizes"].apply(lambda x: len(x))


table["best_val_train_loss_ratio"] = table["best_val_loss"] / table["best_train_loss"]

table["log10(trainable_params)"] = table["trainable_params"].apply(
    lambda x: np.log10(x)
)
table["log10(best_val_loss)"] = table["best_val_loss"].apply(lambda x: np.log10(x))
table["log10(best_train_loss)"] = table["best_train_loss"].apply(lambda x: np.log10(x))
table["log10(learning_rate)"] = table["learning_rate"].apply(lambda x: np.log10(x))
table["log2(batch_size)"] = table["batch_size"].apply(lambda x: np.log2(x))
table["log10(best_step)"] = table["best_step"].apply(lambda x: np.log10(x))
table["log10(trainable_params)"] = table["trainable_params"].apply(
    lambda x: np.log10(x)
)


table

Loading meta from data/logs/laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed/2026-03-04-19-39-39-JCjMEuLN
Loading meta from data/logs/laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed/2026-03-04-19-26-59-DQJPQUgp


,data_path,logs_path,experiment_name,version,batch_size,num_workers,persistent_workers,pin_memory,hidden_layer_sizes,leaky_relu_factor,...,best_train_loss,max_width,depth,best_val_train_loss_ratio,log10(trainable_params),log10(best_val_loss),log10(best_train_loss),log10(learning_rate),log2(batch_size),log10(best_step)
0,data/laser-pulse-shaping-astra-sim-v20+v30-ln-...,data/logs,laser-pulse-shaping-astra-sim-v20+v30-ln-std-e...,2026-03-04-19-39-39-JCjMEuLN,32,4,True,True,"[40, 40, 40, 40, 40]",0.1,...,0.120499,40,5,1.113954,3.916243,-0.87215,-0.919017,-1.0,5.0,4.557206
1,data/laser-pulse-shaping-astra-sim-v20+v30-ln-...,data/logs,laser-pulse-shaping-astra-sim-v20+v30-ln-std-e...,2026-03-04-19-26-59-DQJPQUgp,32,4,True,True,"[40, 40, 40, 40, 40]",0.1,...,NaN,40,5,NaN,NaN,NaN,NaN,-1.0,5.0,NaN


In [5]:
px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log2(batch_size)",
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="best_val_train_loss_ratio",
    color_continuous_scale="RdBu",  # red–blue color scale
    range_color=[0, 2],  # set colorbar limits
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="max_width",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="depth",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

## Best Model Overall


### Paretro front


In [6]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="max_width",
).show()

In [7]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(trainable_params)",
).show()

In [8]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(learning_rate)",
).show()

In [9]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="depth",
).show()

In [10]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(trainable_params)",
).show()

In [11]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(best_step)",
).show()

### Select


Logbook:

- `v11`
  - 2026-01-09-16-13-59-NNYpe2iL:

    `hidden_layers=[50]**4`
    `best_val_loss=0.0017`

    -->`best_val_train_loss_ratio=1.9`<--

    ```python
      def select_best_model():
          df = table.copy()
          return df.iloc[df["log10(best_val_loss)"].argmin()]
    ```

  - 2026-01-09-16-13-58-hGpjScbq:

    -->`hidden_layers=[50]`<--
    `best_val_loss=0.0055`

    ```python
    def select_best_model():
        df = table.copy()
        df = df[df["best_val_train_loss_ratio"] <= 1.2]
        return df.iloc[df["log10(best_val_loss)"].argmin()]
    ```

- `v20`
  - 2026-01-13
  - 2026-02-07


In [12]:
def select_best_model():
    df = table.copy()
    # df = df[df["best_val_train_loss_ratio"] <= 1.2]
    return df.iloc[df["log10(best_val_loss)"].argmin()]


best = select_best_model()
best,

(data_path                                                data/laser-pulse-shaping-astra-sim-v20+v30-ln-...
 logs_path                                                                                        data/logs
 experiment_name                                          laser-pulse-shaping-astra-sim-v20+v30-ln-std-e...
 version                                                                       2026-03-04-19-39-39-JCjMEuLN
 batch_size                                                                                              32
 num_workers                                                                                              4
 persistent_workers                                                                                    True
 pin_memory                                                                                            True
 hidden_layer_sizes                                                                    [40, 40, 40, 40, 40]
 leaky_relu_factor          

In [13]:
best_path = f"{args.logs_path}/{args.experiment_name}/_best"

if os.path.islink(best_path):
    os.unlink(best_path)

os.symlink(best["version"], best_path, target_is_directory=True)

In [14]:
selected_version = best["version"]
selected_version

'2026-03-04-19-39-39-JCjMEuLN'